# Project Checkpoint 1: Dataset Comparison, Selection, and EDA

**Aayush Upadhyay** | UIN: 436000751 | CSCE 676 Data Mining | Spring 2026

---

### Collaboration Declaration

1. **Collaborators**: None
2. **Web Sources**: Kaggle, Ergast Developer API docs, OpenF1 API docs, a couple GitHub repos for MotoGP data
3. **AI Tools**: Used Claude to help brainstorm what EDA to run, boilerplate plotting code, and cleaning up my writing
4. **Citations**: Ergast Developer API (http://ergast.com/mrd/), OpenF1 (https://openf1.org/), MarioJurado/MotoGP GitHub repo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path("data")

---

## (A) Identification of Candidate Datasets

I'm a big Formula 1 and motorcycle racing fan, so I wanted to find datasets in that space that would let me apply the techniques we've been learning in class (frequent itemsets, graph mining, PageRank, large-scale ML) while also giving me room to try something new beyond the course material.

I looked at a bunch of different sources, Kaggle, Zenodo, GitHub repos, and APIs like Ergast and OpenF1. Below are the three datasets I ended up considering.

---

### Dataset 1: Formula 1 World Championship (1950 - 2022)

| Field | Details |
|---|---|
| **Dataset name and source** | Formula 1 World Championship, sourced from the Ergast Developer API. Available on [Kaggle (rohanrao)](https://www.kaggle.com/datasets/rohanrao/formula-1-world-championship-1950-2020) and as a [GitHub mirror](https://github.com/rubenv/ergast-mrd) |
| **Course topic alignment** | Graph mining (build driver-constructor-circuit networks, run PageRank/centrality), frequent itemsets (mine pit stop strategy combos and podium co-occurrences), large-scale ML (predict race outcomes with ensemble methods like random forests and gradient boosted trees) |
| **Potential beyond-course techniques** | Survival analysis for modeling DNF (did not finish) events; time-series forecasting of lap time degradation across stints |
| **Dataset size and structure** | 14 relational CSV tables. Key sizes: ~515K lap time records, ~25K race results, ~8.9K pit stops, ~9.2K qualifying entries, 1079 races, 854 drivers, 211 constructors, 79 circuits. Spans 1950-2022. |
| **Data types** | Numeric IDs, categorical (driver names, constructor names, nationalities, circuit locations), temporal (dates, lap times in ms, pit stop durations), ordinal (grid positions, finishing positions, championship standings) |
| **Target variable(s)** | Multiple depending on the task: finishing position (supervised), DNF/status (binary classification), championship points (regression). For unsupervised tasks like frequent itemsets and graph mining, no explicit target. |
| **Licensing** | Creative Commons Attribution-NonCommercial-ShareAlike 3.0 (CC-BY-NC-SA 3.0). Free for academic/non-commercial use. |

---

### Dataset 2: OpenF1 Telemetry Data (2023 - present)

| Field | Details |
|---|---|
| **Dataset name and source** | OpenF1 real-time and historical telemetry data, available via the [OpenF1 API](https://openf1.org/) |
| **Course topic alignment** | Anomaly detection (flag unusual telemetry readings that precede mechanical failures), clustering (group drivers by driving style based on throttle/brake/speed profiles), large-scale ML (predict position changes from telemetry features) |
| **Potential beyond-course techniques** | Autoencoder-based anomaly detection on multivariate telemetry streams; this is a deep learning approach to finding unusual patterns that simpler threshold methods would miss |
| **Dataset size and structure** | 18 API endpoints. Car telemetry sampled at 3.7 Hz (speed, throttle, brake, RPM, gear) across 20 cars, ~60 laps per race. Roughly millions of rows per race weekend. Also includes lap times, pit stops, tire stints, weather, race control events, team radio. |
| **Data types** | Continuous numeric (speed in km/h, throttle %, brake binary, RPM), temporal (timestamps at sub-second resolution), categorical (driver codes, team names, tire compounds, flag types) |
| **Target variable(s)** | Anomaly labels (unsupervised, would need to derive), position deltas, lap time predictions |
| **Licensing** | Free for historical data access (2023+), non-commercial. Real-time data requires a paid subscription. |

---

### Dataset 3: MotoGP Historical Data (1949 - 2022)

| Field | Details |
|---|---|
| **Dataset name and source** | MotoGP historical race data, compiled from the official MotoGP API via Racing Mike's API. Available on [GitHub (amalsalilan)](https://github.com/amalsalilan/which-nation-excels-in-the-MotoGP) and [GitHub (MarioJurado)](https://github.com/MarioJurado/MotoGP) |
| **Course topic alignment** | Graph mining (rider-team-circuit networks, compute centrality and PageRank across competitive relationships), frequent itemsets (mine podium co-occurrence patterns by manufacturer and nationality) |
| **Potential beyond-course techniques** | Network embedding via node2vec on the rider-team rivalry graph, learning vector representations of riders based on their competitive relationships |
| **Dataset size and structure** | CSV tables covering races, results, riders, constructors, circuits, championships, and world standings across MotoGP, Moto2, and Moto3 categories. 70+ years of data. Smaller than the F1 dataset since fewer data points per race (no lap times or pit stops in the public data). |
| **Data types** | Numeric IDs, categorical (rider names, nationalities, team/constructor names), temporal (race dates, season years), ordinal (finishing positions, championship standings, points) |
| **Target variable(s)** | Championship standings (regression), podium finishes (classification). Mostly unsupervised for graph/itemset tasks. |
| **Licensing** | Publicly available, scraped from the official MotoGP API. No explicit license listed on the GitHub repos, but the data is factual race results (not copyrightable content). |

---

## (B) Comparative Analysis of Datasets

Here I compare the three datasets across the five required dimensions. The goal is to figure out which one gives me the best balance of course technique support, interesting beyond-course work, and practical feasibility.

| Dimension | F1 World Championship (1950-2022) | OpenF1 Telemetry (2023+) | MotoGP Historical (1949-2022) |
|---|---|---|---|
| **Supported data mining tasks** | Frequent itemsets (pit strategy combos, podium co-occurrence), graph mining with PageRank/centrality (driver-constructor-circuit networks), large-scale ML with ensembles (race outcome prediction), survival analysis for DNF prediction (beyond course) | Anomaly detection on telemetry streams (course), clustering of driving styles (course), autoencoder-based anomaly detection (beyond course) | Graph mining on rider-team networks (course), frequent itemsets on podium patterns (course), node2vec network embeddings (beyond course) |
| **Data quality issues** | Pretty clean overall since Ergast is well-maintained and widely used. Some missing lap times for older races (pre-1996). Pit stop data only available from 2012 onward. A few `\N` placeholders for missing values in qualifying times and fastest laps. Sprint results are sparse (only since 2021). | API returns clean structured JSON but you have to deal with rate limits and assembling data across many requests. Telemetry can have gaps if cars go off track or have sensor issues. Only covers 2023 onward so limited historical depth. Weather data can be spotty for some sessions. | Less standardized than F1 data since it's scraped from multiple sources. Some inconsistencies in rider names across eras. No lap-level data in the public version, just final results. The GitHub repos have different table structures that would need reconciling. |
| **Algorithmic feasibility** | Apriori is totally feasible here, the strategy combos per race are a natural basket-like structure and the dataset fits in memory easily. Graph construction is straightforward from the relational tables. Ensemble ML models will have no trouble with ~25K result rows. Survival analysis libraries (lifelines) handle this scale well. | The raw telemetry volume (millions of rows per weekend) would need careful sampling or aggregation to make algorithms work without excessive compute. Autoencoders would benefit from GPU access but aren't strictly necessary at sampled resolution. Clustering 20 drivers per race is very manageable. | Apriori works fine on this data, but the "baskets" are less natural than F1 pit strategies. Graph size is very manageable. The main limitation is that without lap-level data, some analyses (like temporal pattern mining) aren't possible. node2vec runs fine on graphs of this size. |
| **Bias considerations** | Survivorship bias: we only see data for drivers who made it to F1, which skews any analysis of "talent." Era bias is significant since the sport has changed dramatically (car tech, safety, number of races per season). Constructor dominance eras (like Mercedes 2014-2020) can skew pattern mining if not handled carefully. Western/European bias in early decades of the sport. | Recency bias since only 2023+ data exists. The telemetry is objective sensor data so less prone to human bias, but the interpretation of "anomalous" patterns requires domain assumptions. Also, the 2023+ era has specific regulation sets that don't generalize to other eras. | Similar era bias as F1. Strong European/Japanese manufacturer bias in the data. The sport has changed classes and rules many times, making cross-era comparisons tricky. Smaller grids in some eras versus others affect pattern frequencies. |
| **Ethical considerations** | Low risk overall. This is publicly available sports performance data, no personal or sensitive information. The main consideration is that building predictive models could theoretically be used for sports betting, but that's true of any sports dataset. The CC-BY-NC-SA license explicitly restricts commercial use. No power dynamic issues since these are public figures competing in a public sport. | Team radio transcripts could contain sensitive communications, but we'd only use telemetry numbers. Telemetry data could theoretically reveal proprietary team strategies, but OpenF1 only publishes what FIA already broadcasts publicly. No harm to individuals from analyzing car sensor data. | Same low-risk profile as F1. Public sports data about professional athletes. No privacy concerns. The scraped nature of the data means we should be respectful of the API provider's terms, but the data itself is factual results that are publicly known. |

---

## (C) Dataset Selection

**Selected dataset: Formula 1 World Championship (1950 - 2022)**

### Why I picked this one

The F1 World Championship dataset is the strongest choice for a few reasons:

- **It supports the most course techniques natively.** The 14 relational tables give me natural structures for frequent itemsets (pit stop strategy combinations as "baskets"), graph mining (driver-constructor-circuit relationships form a rich multi-relational graph for PageRank and community detection), and large-scale ML (predicting race outcomes with ensemble methods on ~25K results). That covers three major course topics in one dataset.

- **The beyond-course technique is practical and interesting.** Survival analysis for DNF prediction is a real thing teams care about, and Python's `lifelines` library makes it very doable without having to implement anything from scratch. I can also explore time-series forecasting on lap time degradation as a secondary beyond-course technique.

- **The data is clean and well-structured.** Ergast has been maintained for years and is one of the most widely used F1 data sources in the analytics community. The relational structure means joins are straightforward and the data types are consistent.

- **The size is right.** 515K lap times and 25K results is big enough to be interesting but small enough to run everything on a laptop without needing distributed computing or heavy GPU time.

### Trade-offs I'm accepting

- **No text data.** Unlike a Reddit/social media dataset, there's no natural language component here. If I wanted to do text mining or NLP, I'd need to supplement with an external source. But since text mining isn't the focus of my beyond-course technique, this is fine.

- **No real-time telemetry.** The OpenF1 dataset has richer per-lap sensor data (speed, throttle, brake at sub-second intervals), which would be cool for anomaly detection. But the added complexity of API data collection and the limited time range (2023+) make it less practical for this project.

- **Historical data cuts off at 2022.** The Ergast mirror I'm using goes up to 2022. I could supplement with newer data from the jolpica-f1 API if needed later, but for checkpoint 1, what we have is more than enough.